# Assignment Strategy Comparison

Compares **three** inventory placement strategies on a **1 500-aisle warehouse** (25 replicas × 60
aisle types covering every pallet-size / category / handling combination).

| | Strategy | Description |
|---|---|---|
| **A** | Uniform | `_uniform_assignment` — random bin selection (baseline) |
| **B** | Load-Minimizing | `build_load_minimizing_assignment_fn` — greedy L₂ minimisation (best case) |
| **C** | Load-Maximizing | `build_load_maximizing_assignment_fn` — greedy L₂ maximisation (worst case) |

All three simulations process the **same 1 000 pre-generated batches** drawn with lift-weighted
affinity sampling so batch composition reflects co-ordering correlations within each
`(handling, category)` group.  Strategy C gives the congestion ceiling — the gap B→A→C shows
how much layout alone can affect throughput.

| Parameter | Value |
|-----------|-------|
| Aisle config types | 60 (48 pallet + 12 singleton) |
| Pallet sizes covered | small, medium, large, extra_large |
| Total aisles | 1 500 (25 replicas × 60 types) |
| Bin fill target | 85% (~114 750 SKUs) |
| Pickers | 25 |
| Batches | 1 000 |
| Affinity | Sparse (≤ 500 SKUs/group, ~3 M entries) — batch sampling, placement, and lift_sum |

Recovered `LoadParams` are loaded from `recovered_params.json` if present, otherwise
defaults (λ=1, γ=1.5) are used.

In [1]:
import os
import random
import sys
import time as _time
import json as _json
from collections import defaultdict as _dd
from datetime import datetime

_here = os.getcwd()
sys.path.insert(0, os.path.normpath(os.path.join(_here, '..', 'Warehouse')))
sys.path.insert(0, _here)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

from Aisle_Storage import Aisle
from Carton import Carton
from Inventory_Builder import Inventory_Builder, InventoryConfig
from Inventory_Management import Inventory_Manager
from Pick import PickConfig, PickSimulation
from Warehouse_Builder import AisleConfig, Warehouse_Builder, WarehouseConfig
from Workload_Builder import Batch, BatchConfig, Task

from Picking_Analytics import (
    LoadParams,
    build_load_minimizing_assignment_fn,
    build_load_maximizing_assignment_fn,
)
from Picking_Analytics import sum_lift as _sum_lift
from Picking_Data import (
    BatchStats, TaskStats,
    create_run, init_run_db,
    save_batch_stats, load_batch_stats,
    save_task_stats,  load_task_stats,
)
from Simulation_Analytics import (
    extract_batch_stats, extract_task_stats,
    flag_batch_outliers, flag_task_outliers,
)
from Workload import WorkloadParams

print('Imports OK')

Imports OK


In [2]:
_ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
BASE_DIR = os.path.join(_here, f'comparison_{_ts}')
os.makedirs(BASE_DIR, exist_ok=True)
print(f'Base directory : {BASE_DIR}')

# ── Regression configurations ─────────────────────────────────────────────────
# Each entry drives one full warehouse re-build + 1 000-batch simulation.
# 'name' becomes the subfolder under BASE_DIR; omit it to auto-generate from
# coefficient values.  All keys match PickConfig field names.
REGRESSION_CONFIGS = [
    {
        'name'            : 'baseline',
        'pick_weight_coef': 1.1,
        'pick_volume_coef': 1e-3,
        'pick_intercept'  : 1.0,
        'cart_swap_coef'  : 10.0,
    },
    {
        'name'            : 'high_weight',
        'pick_weight_coef': 2.5,
        'pick_volume_coef': 1e-3,
        'pick_intercept'  : 1.0,
        'cart_swap_coef'  : 10.0,
    },
    {
        'name'            : 'high_cart_penalty',
        'pick_weight_coef': 1.1,
        'pick_volume_coef': 1e-3,
        'pick_intercept'  : 1.0,
        'cart_swap_coef'  : 25.0,
    },
    {
        'name'            : 'high_cart_weight_penalty',
        'pick_weight_coef': 2.5,
        'pick_volume_coef': 1e-3,
        'pick_intercept'  : 1.0,
        'cart_swap_coef'  : 25.0,
    },
    {
        'name'            : 'high_cart_weight_volume_penalty',
        'pick_weight_coef': 2.5,
        'pick_volume_coef': 5e-3,
        'pick_intercept'  : 1.0,
        'cart_swap_coef'  : 25.0,
    },
]
print(f'{len(REGRESSION_CONFIGS)} regression config(s):')
for _c in REGRESSION_CONFIGS:
    print(f"  {_c.get('name','unnamed'):22s}  "
          f"w={_c.get('pick_weight_coef',1.1)}  "
          f"v={_c.get('pick_volume_coef',1e-3)}  "
          f"i={_c.get('pick_intercept',1.0)}  "
          f"c={_c.get('cart_swap_coef',10.0)}")

Base directory : c:\Users\myfir\Code and Repos\Inventory_Location_Optimizer\Optimization\comparison_20260528_075855
5 regression config(s):
  baseline                w=1.1  v=0.001  i=1.0  c=10.0
  high_weight             w=2.5  v=0.001  i=1.0  c=10.0
  high_cart_penalty       w=1.1  v=0.001  i=1.0  c=25.0
  high_cart_weight_penalty  w=2.5  v=0.001  i=1.0  c=25.0
  high_cart_weight_volume_penalty  w=2.5  v=0.005  i=1.0  c=25.0


## Configuration

In [3]:
SEED_WORLD   = 42
SEED_BATCHES = 1337
N_BATCHES    = 1_000
K_PICKERS    = 25

_CATEGORIES  = ['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']
_ALL_SIZES   = ['small', 'medium', 'large', 'extra_large']

_CONV_SIZES  = ['small', 'medium', 'large']
_CONV_PROBS  = [0.25, 0.50, 0.25]
_NCONV_SIZES = ['medium', 'large', 'extra_large']
_NCONV_PROBS = [0.20, 0.50, 0.30]

# Bin dimensions: 25 bayX × 20 bayY = 500 bins per aisle
_AISLE_CFGS = []
for _size in _ALL_SIZES:
    for _cat in _CATEGORIES:
        _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'pallet', 25, 20, [_size], None))
        _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'pallet', 25, 20, [_size], None))
for _cat in _CATEGORIES:
    _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'singleton', 25, 20, _CONV_SIZES,  _CONV_PROBS))
    _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'singleton', 25, 20, _NCONV_SIZES, _NCONV_PROBS))

_N_TYPES      = len(_AISLE_CFGS)   # 60
_REPLICAS     = 3
_TOTAL_AISLES = _N_TYPES * _REPLICAS   # 180

WAREHOUSE_CFG = WarehouseConfig(
    total_aisles  = _TOTAL_AISLES,
    aisle_splits  = [1 / _N_TYPES] * _N_TYPES,
    aisle_configs = _AISLE_CFGS,
)

_A_COL      = '#5b9bd5'   # blue  — Uniform
_B_COL      = '#f4a030'   # amber — Load-Minimizing
_C_COL      = '#70ad47'   # green — Load-Maximizing
_TRAVEL_COL = '#a9a9a9'   # grey  — traveling segment in stacked bar

print(f'Aisle config types : {_N_TYPES}  (48 pallet + 12 singleton)')
print(f'Bins per aisle     : 500  (25 bayX × 20 bayY)')
print(f'Total aisles       : {_TOTAL_AISLES}  ({_REPLICAS} replicas × {_N_TYPES} types)')
print(f'Pickers            : {K_PICKERS}')
print(f'Batches per config : {N_BATCHES:,}')

Aisle config types : 60  (48 pallet + 12 singleton)
Bins per aisle     : 500  (25 bayX × 20 bayY)
Total aisles       : 180  (3 replicas × 60 types)
Pickers            : 25
Batches per config : 1,000


## Build Inventory & Affinity

A probe build measures the exact bin count, then inventory is sized to 85% fill.

A **sparse affinity** (≤ 500 SKUs per `(handling, category)` group, ~3 M entries) is built
from the inventory before any warehouse is stocked.  It is used for:
- **Batch sampling** — lift-weighted selection draws co-ordering correlated SKUs; the
  partner-map is cached on first call so the O(|affinity|) build is paid only once
- `build_load_minimizing_assignment_fn` — co-locating high-affinity SKUs in warehouse B
- `extract_task_stats` — computing `lift_sum` on each task

In [4]:
_FILL_TARGET     = 0.85
_BATCH_MEAN_FRAC = 0.25
_BATCH_STD_FRAC  = 0.03
_MAX_PER_GROUP   = 500   # SKUs per (handling, category) group in sparse affinity

# ── Probe to measure exact bin count ─────────────────────────────────────────
Carton.next_sku     = 1
Aisle.next_aisle_id = 1
random.seed(SEED_WORLD)
_probe     = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()
TOTAL_BINS = len(_probe.bins)
N_SKUS     = int(_FILL_TARGET * TOTAL_BINS)
del _probe

INV_CFG = InventoryConfig(
    num_skus           = N_SKUS,
    handling_splits    = [0.5, 0.5],
    category_splits    = [1 / 6] * 6,
    singleton_fraction = 0.5,
)

Carton.next_sku = 1
random.seed(SEED_WORLD + 10)
inventory = Inventory_Builder().from_config(INV_CFG).build()

# ── Sparse affinity from inventory (pre-placement) ────────────────────────────
# Groups by (handling, category) — only co-located SKUs can have correlated demand.
# Capped at _MAX_PER_GROUP per group; 12 groups × C(500,2) × 2 ≈ 3 M entries.
print('Building sparse affinity...', end=' ', flush=True)
t0 = _time.perf_counter()
AFFINITY = inventory.affinity_matrix(max_per_group=_MAX_PER_GROUP)
print(f'{len(AFFINITY):,} entries  ({_time.perf_counter() - t0:.2f}s)')

BATCH_CFG = BatchConfig(
    inventory_size = N_SKUS,
    mean_fraction  = _BATCH_MEAN_FRAC,
    std_fraction   = _BATCH_STD_FRAC,
)

# ── Load recovered LoadParams or use defaults ─────────────────────────────────
_param_path = os.path.join(_here, 'recovered_params.json')
if os.path.exists(_param_path):
    _p = _json.load(open(_param_path))
    LOAD_PARAMS = LoadParams(lambda_=_p['lambda_'], k=1.0, gamma=_p['gamma'])
    print(f'Recovered params  λ={LOAD_PARAMS.lambda_:.4f}  γ={LOAD_PARAMS.gamma:.4f}')
else:
    LOAD_PARAMS = LoadParams(lambda_=1.0, k=1.0, gamma=1.5)
    print('recovered_params.json not found — using defaults (λ=1.0  γ=1.5)')

_skus_per_aisle = N_SKUS / _TOTAL_AISLES
_expected_picks = _skus_per_aisle * _BATCH_MEAN_FRAC
print(f'\nWarehouse bins  : {TOTAL_BINS:,}')
print(f'SKU catalog     : {N_SKUS:,}  ({_FILL_TARGET:.0%} fill target)')
print(f'SKUs / aisle    : ~{_skus_per_aisle:.0f}')
print(f'Avg batch size  : ~{int(N_SKUS * _BATCH_MEAN_FRAC):,} SKUs  ({_BATCH_MEAN_FRAC:.0%})')
print(f'Expected picks/aisle/task : ~{_expected_picks:.0f}')

Building sparse affinity... 2,994,000 entries  (0.83s)
Recovered params  λ=0.6603  γ=-0.1356

Warehouse bins  : 90,000
SKU catalog     : 76,500  (85% fill target)
SKUs / aisle    : ~425
Avg batch size  : ~19,125 SKUs  (25%)
Expected picks/aisle/task : ~106


## Build Warehouses A, B, and C

In [5]:
# ── Warehouse A: uniform (random) assignment — built once, shared across configs ─
Aisle.next_aisle_id = 1
random.seed(SEED_WORLD)
warehouse_A = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()

random.seed(SEED_WORLD + 100)
manager_A   = Inventory_Manager(warehouse_A)
manager_A.enqueue_all(inventory.cartons, quantity=1)

placed_A = len(manager_A.unavailable)
print(f'Warehouse A (uniform) : {placed_A:,} / {TOTAL_BINS:,} bins filled  ({placed_A / TOTAL_BINS:.1%})')

# ── Aisle metadata maps (physical layout identical across A, B, C) ─────────────
AISLE_SIZE_MAP     = {a.aisle_id: a.storage_size  for a in warehouse_A.aisles}
AISLE_UNITTYPE_MAP = {a.aisle_id: a.unit_type     for a in warehouse_A.aisles}
AISLE_HANDLING_MAP = {a.aisle_id: a.handling_type for a in warehouse_A.aisles}

Warehouse A (uniform) : 55,449 / 90,000 bins filled  (61.6%)


In [ ]:
_SIZE_ORDER  = ['small', 'medium', 'large', 'extra_large']
_SIZE_LABELS = ['Small', 'Medium', 'Large', 'Extra-Large']

def _bdf(stats):
    return pd.DataFrame([{
        'batch_id'              : s.batch_id,
        'duration'              : s.duration,
        'num_tasks'             : s.num_tasks,
        'total_items'           : s.total_items,
        'completion_rate'       : s.total_items / s.duration if s.duration > 0 else 0.0,
        'avg_concurrent_pickers': s.avg_concurrent_pickers,
        'picking_pct'           : s.picking_pct   * 100,
        'traveling_pct'         : s.traveling_pct * 100,
    } for s in stats])

def _tdf(stats):
    return pd.DataFrame([{
        'batch_id'   : s.batch_id,
        'aisle_id'   : s.aisle_id,
        'duration'   : s.duration,
        'W_a'        : s.W_a,
        'lift_sum'   : s.lift_sum,
        'num_bins'   : s.num_bins_visited,
        'total_items': s.total_items,
        'pallet_size': AISLE_SIZE_MAP.get(s.aisle_id),
        'unit_type'  : AISLE_UNITTYPE_MAP.get(s.aisle_id),
        'handling'   : AISLE_HANDLING_MAP.get(s.aisle_id),
    } for s in stats])

def _pallet_df(df):
    d = df[(df['unit_type'] == 'pallet') & (df['pallet_size'].notna())].copy()
    d['pallet_size'] = pd.Categorical(d['pallet_size'], categories=_SIZE_ORDER, ordered=True)
    return d

def _aisle_lift_sums(wh):
    by_aisle = _dd(list)
    for b in wh.bins:
        if b.storage is not None:
            by_aisle[b.location[0]].append(b.storage.carton.sku)
    return {aid: _sum_lift(skus, AFFINITY) for aid, skus in by_aisle.items()}

def _roll(df, col, win=50):
    return df.sort_values('batch_id')[col].rolling(win, min_periods=1).mean().values

print('Helper functions defined.')

Helper functions defined.


: 

## Regression Config Loop

For each entry in `REGRESSION_CONFIGS` the notebook:
1. Builds `PICK_CFG` / `WP` from the config's coefficients and creates a subfolder under `BASE_DIR`
2. Writes `config.json` with the full parameter set for reproducibility
3. Rebuilds **Warehouses B and C** using those coefficients (A is fixed as the uniform baseline)
4. Runs **1 000 batches** sequentially (Create → Tasks → Simulate)
5. Saves the **database** and all **plots** into the config's subfolder

In [ ]:
_CHECKPOINT = 100
_WIN        = 50

for _cfg in REGRESSION_CONFIGS:
    # ── Per-config setup ──────────────────────────────────────────────────────
    _name = _cfg.get('name') or (
        f"w{_cfg.get('pick_weight_coef',1.1)}_v{_cfg.get('pick_volume_coef',1e-3)}"
        f"_i{_cfg.get('pick_intercept',1.0)}_c{_cfg.get('cart_swap_coef',10.0)}"
    )
    PICK_CFG = PickConfig(
        num_pickers      = K_PICKERS,
        x_move_time      = _cfg.get('x_move_time',      1.0),
        y_move_time      = _cfg.get('y_move_time',      0.5),
        pick_intercept   = _cfg.get('pick_intercept',   1.0),
        pick_weight_coef = _cfg.get('pick_weight_coef', 1.1),
        pick_volume_coef = _cfg.get('pick_volume_coef', 1e-3),
        cart_swap_coef   = _cfg.get('cart_swap_coef',   10.0),
    )
    WP      = WorkloadParams.from_pick_config(PICK_CFG)
    RUN_DIR = os.path.join(BASE_DIR, _name)
    os.makedirs(RUN_DIR, exist_ok=True)
    DB_PATH = os.path.join(RUN_DIR, 'sim.db')

    # ── Write config.json ─────────────────────────────────────────────────────
    _config_record = {
        'name'            : _name,
        'pick_weight_coef': PICK_CFG.pick_weight_coef,
        'pick_volume_coef': PICK_CFG.pick_volume_coef,
        'pick_intercept'  : PICK_CFG.pick_intercept,
        'cart_swap_coef'  : PICK_CFG.cart_swap_coef,
        'x_move_time'     : PICK_CFG.x_move_time,
        'y_move_time'     : PICK_CFG.y_move_time,
        'num_pickers'     : PICK_CFG.num_pickers,
        'total_aisles'    : _TOTAL_AISLES,
        'bins_per_aisle'  : 500,
        'fill_target'     : _FILL_TARGET,
        'batch_mean_frac' : _BATCH_MEAN_FRAC,
        'n_batches'       : N_BATCHES,
        'seed_world'      : SEED_WORLD,
        'seed_batches'    : SEED_BATCHES,
    }
    with open(os.path.join(RUN_DIR, 'config.json'), 'w') as _f:
        _json.dump(_config_record, _f, indent=2)

    print(f'\n{"="*68}')
    print(f'  Config : {_name}')
    print(f'  Dir    : {RUN_DIR}')
    print(f'  w={PICK_CFG.pick_weight_coef}  v={PICK_CFG.pick_volume_coef}  '
          f'i={PICK_CFG.pick_intercept}  c={PICK_CFG.cart_swap_coef}')
    print(f'{"="*68}')

    # ── Build Warehouses B and C (re-optimised with this config's coefficients) ─
    Aisle.next_aisle_id = 1
    random.seed(SEED_WORLD)
    warehouse_B = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()
    print('  Stocking B (load-min)...', end=' ', flush=True)
    t0 = _time.perf_counter()
    random.seed(SEED_WORLD + 200)
    manager_B = Inventory_Manager(
        warehouse_B,
        assignment_fn=build_load_minimizing_assignment_fn(LOAD_PARAMS, AFFINITY, WP),
    )
    manager_B.enqueue_all(inventory.cartons, quantity=1)
    print(f'{_time.perf_counter() - t0:.1f}s  ({len(manager_B.unavailable):,} bins filled)')

    Aisle.next_aisle_id = 1
    random.seed(SEED_WORLD)
    warehouse_C = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()
    print('  Stocking C (load-max)...', end=' ', flush=True)
    t0 = _time.perf_counter()
    random.seed(SEED_WORLD + 300)
    manager_C = Inventory_Manager(
        warehouse_C,
        assignment_fn=build_load_maximizing_assignment_fn(LOAD_PARAMS, AFFINITY, WP),
    )
    manager_C.enqueue_all(inventory.cartons, quantity=1)
    print(f'{_time.perf_counter() - t0:.1f}s  ({len(manager_C.unavailable):,} bins filled)')

    # ── Lift-sum distribution ─────────────────────────────────────────────────
    _la = np.array([_aisle_lift_sums(warehouse_A).get(a, 0.0) for a in range(1, _TOTAL_AISLES + 1)])
    _lb = np.array([_aisle_lift_sums(warehouse_B).get(a, 0.0) for a in range(1, _TOTAL_AISLES + 1)])
    _lc = np.array([_aisle_lift_sums(warehouse_C).get(a, 0.0) for a in range(1, _TOTAL_AISLES + 1)])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    fig.suptitle(f'Per-Aisle Lift Sum after Assignment  [{_name}]', fontsize=12, fontweight='bold')
    for _v, _lbl, _c in [(_la[_la>0], 'Uniform (A)', _A_COL),
                          (_lb[_lb>0], 'Load-Min (B)', _B_COL),
                          (_lc[_lc>0], 'Load-Max (C)', _C_COL)]:
        axes[0].hist(_v, bins=50, color=_c, alpha=0.50, edgecolor='white', label=f'{_lbl}  μ={_v.mean():.1f}')
        axes[0].axvline(_v.mean(), color=_c, lw=2, linestyle='--')
    axes[0].set_title('Lift sum distribution  (aisles with lift > 0)', fontsize=10)
    axes[0].set_xlabel('Σ lift');  axes[0].set_ylabel('Aisle count')
    axes[0].legend(fontsize=9);   axes[0].grid(axis='y', alpha=0.3)
    for _v, _lbl, _c in [(np.sort(_la), 'Uniform (A)',   _A_COL),
                          (np.sort(_lb), 'Load-Min (B)', _B_COL),
                          (np.sort(_lc), 'Load-Max (C)', _C_COL)]:
        axes[1].plot(_v, np.arange(1, len(_v)+1) / len(_v), color=_c, lw=2, label=_lbl)
    axes[1].set_xlabel('Per-aisle lift sum');  axes[1].set_ylabel('Cumulative fraction')
    axes[1].set_title('CDF of per-aisle lift sum', fontsize=10)
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    axes[1].legend(fontsize=9);  axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'lift_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── DB init ───────────────────────────────────────────────────────────────
    init_run_db(DB_PATH)
    RUN_A = create_run(DB_PATH, 'uniform_assignment')
    RUN_B = create_run(DB_PATH, 'load_minimizing_assignment')
    RUN_C = create_run(DB_PATH, 'load_maximizing_assignment')

    # ── Simulation loop  (Create → Tasks → Simulate) ──────────────────────────
    all_bs_A: list = [];  all_bs_B: list = [];  all_bs_C: list = []
    all_ts_A: list = [];  all_ts_B: list = [];  all_ts_C: list = []
    _pb_A: list = [];     _pb_B: list = [];     _pb_C: list = []
    _pt_A: list = [];     _pt_B: list = [];     _pt_C: list = []

    random.seed(SEED_BATCHES)
    t_loop  = _time.perf_counter()
    skipped = 0

    for _i in range(N_BATCHES):
        _batch = Batch(BATCH_CFG, inventory, affinity=AFFINITY)
        _ta = Task.from_batch(_batch, warehouse_A)
        _tb = Task.from_batch(_batch, warehouse_B)
        _tc = Task.from_batch(_batch, warehouse_C)

        if not _ta or not _tb or not _tc:
            skipped += 1
            continue

        _ea = PickSimulation(_ta, PICK_CFG).run()
        _eb = PickSimulation(_tb, PICK_CFG).run()
        _ec = PickSimulation(_tc, PICK_CFG).run()

        _bsA = extract_batch_stats(_ea, batch_id=_i, k_pickers=K_PICKERS, run_id=RUN_A)
        _bsB = extract_batch_stats(_eb, batch_id=_i, k_pickers=K_PICKERS, run_id=RUN_B)
        _bsC = extract_batch_stats(_ec, batch_id=_i, k_pickers=K_PICKERS, run_id=RUN_C)
        _tsA = extract_task_stats(_ea, _ta, batch_id=_i, affinity=AFFINITY, wp=WP, run_id=RUN_A)
        _tsB = extract_task_stats(_eb, _tb, batch_id=_i, affinity=AFFINITY, wp=WP, run_id=RUN_B)
        _tsC = extract_task_stats(_ec, _tc, batch_id=_i, affinity=AFFINITY, wp=WP, run_id=RUN_C)

        _pb_A.append(_bsA);  all_bs_A.append(_bsA)
        _pb_B.append(_bsB);  all_bs_B.append(_bsB)
        _pb_C.append(_bsC);  all_bs_C.append(_bsC)
        _pt_A.extend(_tsA);  all_ts_A.extend(_tsA)
        _pt_B.extend(_tsB);  all_ts_B.extend(_tsB)
        _pt_C.extend(_tsC);  all_ts_C.extend(_tsC)

        if len(_pb_A) >= _CHECKPOINT:
            save_batch_stats(DB_PATH, RUN_A, _pb_A);  save_batch_stats(DB_PATH, RUN_B, _pb_B);  save_batch_stats(DB_PATH, RUN_C, _pb_C)
            save_task_stats( DB_PATH, RUN_A, _pt_A);  save_task_stats( DB_PATH, RUN_B, _pt_B);  save_task_stats( DB_PATH, RUN_C, _pt_C)
            _rate = (_i + 1) / (_time.perf_counter() - t_loop)
            print(f'  Batch {_i+1:4d}/{N_BATCHES}  '
                  f'A={_bsA.duration:.0f}  B={_bsB.duration:.0f}  C={_bsC.duration:.0f}  '
                  f'{_rate:.1f}/s', flush=True)
            _pb_A.clear(); _pb_B.clear(); _pb_C.clear()
            _pt_A.clear(); _pt_B.clear(); _pt_C.clear()

    for _run, _pb, _pt in [(RUN_A, _pb_A, _pt_A), (RUN_B, _pb_B, _pt_B), (RUN_C, _pb_C, _pt_C)]:
        if _pb:
            save_batch_stats(DB_PATH, _run, _pb)
            save_task_stats( DB_PATH, _run, _pt)
    print(f'  Done  {N_BATCHES - skipped} triplets in {_time.perf_counter() - t_loop:.1f}s  ({skipped} skipped)')

    # ── Load data ─────────────────────────────────────────────────────────────
    bs_fA = flag_batch_outliers(load_batch_stats(DB_PATH, RUN_A))
    bs_fB = flag_batch_outliers(load_batch_stats(DB_PATH, RUN_B))
    bs_fC = flag_batch_outliers(load_batch_stats(DB_PATH, RUN_C))
    ts_fA = flag_task_outliers(load_task_stats(DB_PATH, RUN_A))
    ts_fB = flag_task_outliers(load_task_stats(DB_PATH, RUN_B))
    ts_fC = flag_task_outliers(load_task_stats(DB_PATH, RUN_C))
    df_bA = _bdf([s for s in bs_fA if not s.is_outlier])
    df_bB = _bdf([s for s in bs_fB if not s.is_outlier])
    df_bC = _bdf([s for s in bs_fC if not s.is_outlier])
    df_tA = _tdf([s for s in ts_fA if not s.is_outlier])
    df_tB = _tdf([s for s in ts_fB if not s.is_outlier])
    df_tC = _tdf([s for s in ts_fC if not s.is_outlier])

    # ── Summary stats → CSV ───────────────────────────────────────────────────
    _BCOLS = ['duration', 'completion_rate', 'avg_concurrent_pickers', 'picking_pct', 'traveling_pct']
    _TCOLS = ['duration', 'W_a', 'lift_sum', 'num_bins']
    _summ_b = pd.concat(
        [df_bA[_BCOLS].agg(['mean','median','std']).T,
         df_bB[_BCOLS].agg(['mean','median','std']).T,
         df_bC[_BCOLS].agg(['mean','median','std']).T],
        axis=1, keys=['Uniform (A)', 'Load-Min (B)', 'Load-Max (C)']).round(3)
    _summ_t = pd.concat(
        [df_tA[_TCOLS].agg(['mean','median','std']).T,
         df_tB[_TCOLS].agg(['mean','median','std']).T,
         df_tC[_TCOLS].agg(['mean','median','std']).T],
        axis=1, keys=['Uniform (A)', 'Load-Min (B)', 'Load-Max (C)']).round(3)
    _summ_b.to_csv(os.path.join(RUN_DIR, 'summary_batch.csv'))
    _summ_t.to_csv(os.path.join(RUN_DIR, 'summary_task.csv'))
    display(_summ_b);  display(_summ_t)

    # ── Plot 1: Batch Duration ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    fig.suptitle(f'Batch Completion Time  [{_name}]', fontsize=13, fontweight='bold')
    for ax, data, label, color in [
        (axes[0], df_bA['duration'].values, 'A — Uniform',         _A_COL),
        (axes[1], df_bB['duration'].values, 'B — Load-Minimizing', _B_COL),
        (axes[2], df_bC['duration'].values, 'C — Load-Maximizing', _C_COL),
    ]:
        ax.hist(data, bins=40, color=color, alpha=0.65, edgecolor='white')
        if len(data) > 1:
            _kde = gaussian_kde(data, bw_method='silverman')
            _xs  = np.linspace(data.min(), data.max(), 400)
            ax.plot(_xs, _kde(_xs) * len(data) * (data.max() - data.min()) / 40, color=color, lw=2)
        ax.axvline(data.mean(),     color='red',    lw=1.5, linestyle='--', label=f'Mean   {data.mean():.1f}')
        ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',  label=f'Median {np.median(data):.1f}')
        ax.set_xlabel('Batch duration  (sim time units)', fontsize=10);  ax.set_ylabel('Count', fontsize=10)
        ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
        ax.legend(fontsize=8);  ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot1_batch_duration.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 2: Task Duration ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    fig.suptitle(f'Task (Aisle) Completion Time  [{_name}]', fontsize=13, fontweight='bold')
    for ax, data, label, color in [
        (axes[0], df_tA['duration'].values, 'A — Uniform',         _A_COL),
        (axes[1], df_tB['duration'].values, 'B — Load-Minimizing', _B_COL),
        (axes[2], df_tC['duration'].values, 'C — Load-Maximizing', _C_COL),
    ]:
        ax.hist(data, bins=50, color=color, alpha=0.65, edgecolor='white')
        if len(data) > 1:
            _kde = gaussian_kde(data, bw_method='silverman')
            _xs  = np.linspace(data.min(), data.max(), 400)
            ax.plot(_xs, _kde(_xs) * len(data) * (data.max() - data.min()) / 50, color=color, lw=2)
        ax.axvline(data.mean(),     color='red',    lw=1.5, linestyle='--', label=f'Mean   {data.mean():.1f}')
        ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',  label=f'Median {np.median(data):.1f}')
        ax.set_xlabel('Task duration  (sim time units)', fontsize=10);  ax.set_ylabel('Count', fontsize=10)
        ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
        ax.legend(fontsize=8);  ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot2_task_duration.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 3: Completion Rate ───────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
    fig.suptitle(f'Batch Completion Rate  (rolling {_WIN}-batch window)  [{_name}]', fontsize=13, fontweight='bold')
    for ax, col, ylabel, title in [
        (ax1, 'completion_rate', 'Items / time unit',    'Throughput rate'),
        (ax2, 'duration',        'Duration (time units)', 'Batch completion time'),
    ]:
        ax.plot(df_bA.sort_values('batch_id')['batch_id'].values, _roll(df_bA, col, _WIN), color=_A_COL, lw=2, label='Uniform (A)')
        ax.plot(df_bB.sort_values('batch_id')['batch_id'].values, _roll(df_bB, col, _WIN), color=_B_COL, lw=2, label='Load-Min (B)')
        ax.plot(df_bC.sort_values('batch_id')['batch_id'].values, _roll(df_bC, col, _WIN), color=_C_COL, lw=2, label='Load-Max (C)')
        ax.set_ylabel(ylabel, fontsize=10);  ax.set_title(title, fontsize=10)
        ax.legend(fontsize=9);  ax.grid(alpha=0.3)
    ax2.set_xlabel('Batch ID', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot3_completion_rate.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 4: Picker Concurrency ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    fig.suptitle(f'Picker Concurrency  [{_name}]', fontsize=12, fontweight='bold')
    for ax, data, label, color in [
        (axes[0], df_bA['avg_concurrent_pickers'].values, 'A — Uniform',         _A_COL),
        (axes[1], df_bB['avg_concurrent_pickers'].values, 'B — Load-Minimizing', _B_COL),
        (axes[2], df_bC['avg_concurrent_pickers'].values, 'C — Load-Maximizing', _C_COL),
    ]:
        ax.hist(data, bins=35, color=color, alpha=0.70, edgecolor='white')
        if len(data) > 1 and data.max() > data.min():
            _kde = gaussian_kde(data, bw_method='silverman')
            _xs  = np.linspace(data.min(), data.max(), 400)
            ax.plot(_xs, _kde(_xs) * len(data) * (data.max() - data.min()) / 35, color=color, lw=2)
        ax.axvline(data.mean(),     color='red',    lw=1.5, linestyle='--', label=f'Mean   {data.mean():.2f}')
        ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',  label=f'Median {np.median(data):.2f}')
        ax.axvline(K_PICKERS, color='grey', lw=1.0, linestyle='-.', label=f'Max ({K_PICKERS})')
        ax.set_xlabel('Avg concurrent pickers', fontsize=10);  ax.set_ylabel('Count', fontsize=10)
        ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
        ax.legend(fontsize=8);  ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot4_picker_concurrency.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 5: Picker Utilisation ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    fig.suptitle(f'Picker Utilisation Breakdown  [{_name}]', fontsize=13, fontweight='bold')
    _bp = axes[0].boxplot(
        [df_bA['picking_pct'].values,   df_bB['picking_pct'].values,   df_bC['picking_pct'].values,
         df_bA['traveling_pct'].values, df_bB['traveling_pct'].values, df_bC['traveling_pct'].values],
        labels=['Pick A', 'Pick B', 'Pick C', 'Travel A', 'Travel B', 'Travel C'],
        patch_artist=True, medianprops=dict(color='black', lw=2),
    )
    for _patch, _c in zip(_bp['boxes'], [_A_COL, _B_COL, _C_COL, _A_COL, _B_COL, _C_COL]):
        _patch.set_facecolor(_c);  _patch.set_alpha(0.7)
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
    axes[0].set_title('Picking vs Traveling %', fontsize=10);  axes[0].grid(axis='y', alpha=0.3)
    for _dfb, _c, _lbl in [(df_bA, _A_COL, 'Uniform'), (df_bB, _B_COL, 'Load-Min'), (df_bC, _C_COL, 'Load-Max')]:
        axes[1].hist(_dfb['picking_pct'].values, bins=30, color=_c, alpha=0.55, edgecolor='white',
                     label=f'{_lbl}  μ={_dfb["picking_pct"].mean():.1f}%')
    axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
    axes[1].set_xlabel('Picking %', fontsize=10);  axes[1].set_ylabel('Count', fontsize=10)
    axes[1].set_title('Picking % — overlaid', fontsize=10)
    axes[1].legend(fontsize=8);  axes[1].grid(axis='y', alpha=0.3)
    _x  = np.arange(3)
    _pk = [df_bA['picking_pct'].mean(),   df_bB['picking_pct'].mean(),   df_bC['picking_pct'].mean()]
    _tr = [df_bA['traveling_pct'].mean(), df_bB['traveling_pct'].mean(), df_bC['traveling_pct'].mean()]
    axes[2].bar(_x, _pk, width=0.5, color=[_A_COL, _B_COL, _C_COL], alpha=0.85, label='Picking')
    axes[2].bar(_x, _tr, width=0.5, bottom=_pk, color=_TRAVEL_COL, alpha=0.85, label='Traveling')
    axes[2].set_xticks(_x);  axes[2].set_xticklabels(['Uniform (A)', 'Load-Min (B)', 'Load-Max (C)'])
    axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
    axes[2].set_ylabel('Mean fraction (%)', fontsize=10);  axes[2].set_title('Aggregate mean split', fontsize=10)
    axes[2].legend(fontsize=8);  axes[2].grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot5_picker_utilisation.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 6a: Pallet Size Analysis ─────────────────────────────────────────
    dp_A = _pallet_df(df_tA);  dp_B = _pallet_df(df_tB);  dp_C = _pallet_df(df_tC)
    _mdA = [dp_A[dp_A['pallet_size']==s]['duration'].mean() for s in _SIZE_ORDER]
    _mdB = [dp_B[dp_B['pallet_size']==s]['duration'].mean() for s in _SIZE_ORDER]
    _mdC = [dp_C[dp_C['pallet_size']==s]['duration'].mean() for s in _SIZE_ORDER]
    _x2  = np.arange(len(_SIZE_ORDER));  _w2 = 0.25
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle(f'Pallet-Aisle Task Analysis by Storage Size  [{_name}]', fontsize=13, fontweight='bold')
    axes[0].bar(_x2 - _w2, _mdA, width=_w2, color=_A_COL, alpha=0.85, label='Uniform (A)')
    axes[0].bar(_x2,        _mdB, width=_w2, color=_B_COL, alpha=0.85, label='Load-Min (B)')
    axes[0].bar(_x2 + _w2, _mdC, width=_w2, color=_C_COL, alpha=0.85, label='Load-Max (C)')
    axes[0].set_xticks(_x2);  axes[0].set_xticklabels(_SIZE_LABELS)
    axes[0].set_ylabel('Mean task duration', fontsize=10);  axes[0].set_title('Mean task duration per pallet size', fontsize=10)
    axes[0].legend(fontsize=9);  axes[0].grid(axis='y', alpha=0.3)
    _dB2 = [(b-a)/abs(a)*100 if a else 0 for a,b in zip(_mdA,_mdB)]
    _dC2 = [(c-a)/abs(a)*100 if a else 0 for a,c in zip(_mdA,_mdC)]
    axes[1].bar(_x2 - _w2/2, _dB2, width=_w2, color=[_B_COL if d<0 else '#c00000' for d in _dB2], alpha=0.85, label='B vs A')
    axes[1].bar(_x2 + _w2/2, _dC2, width=_w2, color=[_C_COL if d>0 else '#c00000' for d in _dC2], alpha=0.85, label='C vs A')
    axes[1].axhline(0, color='black', lw=1)
    for _j, (dB, dC) in enumerate(zip(_dB2, _dC2)):
        axes[1].text(_j - _w2/2, dB + (0.3 if dB>=0 else -0.6), f'{dB:.1f}%', ha='center', fontsize=8)
        axes[1].text(_j + _w2/2, dC + (0.3 if dC>=0 else -0.6), f'{dC:.1f}%', ha='center', fontsize=8)
    axes[1].set_xticks(_x2);  axes[1].set_xticklabels(_SIZE_LABELS)
    axes[1].set_ylabel('Δ (X − A) / A  %', fontsize=10);  axes[1].set_title('Duration delta per size', fontsize=10)
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
    axes[1].legend(fontsize=8);  axes[1].grid(axis='y', alpha=0.3)
    _SCOLS = ['#9dc3e6', '#5b9bd5', '#2e75b6', '#1f4e79']
    for _sz, _sc, _sl in zip(_SIZE_ORDER, _SCOLS, _SIZE_LABELS):
        _vA = dp_A[dp_A['pallet_size']==_sz]['duration'].values
        _vB = dp_B[dp_B['pallet_size']==_sz]['duration'].values
        _vC = dp_C[dp_C['pallet_size']==_sz]['duration'].values
        _lo = min(_vA.min(),_vB.min(),_vC.min());  _hi = max(_vA.max(),_vB.max(),_vC.max())
        _xs = np.linspace(_lo, _hi, 300)
        if len(_vA)>1: axes[2].plot(_xs, gaussian_kde(_vA,'silverman')(_xs), color=_sc, lw=2,   ls='-',  label=f'{_sl} A')
        if len(_vB)>1: axes[2].plot(_xs, gaussian_kde(_vB,'silverman')(_xs), color=_sc, lw=2,   ls='--', label=f'{_sl} B')
        if len(_vC)>1: axes[2].plot(_xs, gaussian_kde(_vC,'silverman')(_xs), color=_sc, lw=1.5, ls=':',  label=f'{_sl} C')
    axes[2].set_xlabel('Task duration', fontsize=10);  axes[2].set_ylabel('Density', fontsize=10)
    axes[2].set_title('KDE: solid=A, dashed=B, dot=C', fontsize=10)
    axes[2].legend(fontsize=6, ncol=3);  axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot6a_pallet_size.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 6b: Handling Breakdown ───────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Mean Task Duration per Pallet Size × Handling  [{_name}]', fontsize=12, fontweight='bold')
    for ax, _h in [(axes[0], 'conveyable'), (axes[1], 'non-conveyable')]:
        _mA = [dp_A[(dp_A['pallet_size']==s)&(dp_A['handling']==_h)]['duration'].mean() for s in _SIZE_ORDER]
        _mB = [dp_B[(dp_B['pallet_size']==s)&(dp_B['handling']==_h)]['duration'].mean() for s in _SIZE_ORDER]
        _mC = [dp_C[(dp_C['pallet_size']==s)&(dp_C['handling']==_h)]['duration'].mean() for s in _SIZE_ORDER]
        ax.bar(_x2 - _w2, _mA, width=_w2, color=_A_COL, alpha=0.85, label='Uniform (A)')
        ax.bar(_x2,        _mB, width=_w2, color=_B_COL, alpha=0.85, label='Load-Min (B)')
        ax.bar(_x2 + _w2, _mC, width=_w2, color=_C_COL, alpha=0.85, label='Load-Max (C)')
        ax.set_xticks(_x2);  ax.set_xticklabels(_SIZE_LABELS)
        ax.set_title(f'{_h.capitalize()} pallet aisles', fontsize=10)
        ax.set_ylabel('Mean task duration', fontsize=10)
        ax.legend(fontsize=9);  ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot6b_pallet_size_handling.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 7: Per-Aisle ─────────────────────────────────────────────────────
    _acmp = pd.concat([
        df_tA.groupby('aisle_id')['duration'].mean().rename('A'),
        df_tB.groupby('aisle_id')['duration'].mean().rename('B'),
        df_tC.groupby('aisle_id')['duration'].mean().rename('C'),
    ], axis=1).dropna()
    _acmp['dB'] = (_acmp['B'] - _acmp['A']) / _acmp['A'].abs() * 100
    _acmp['dC'] = (_acmp['C'] - _acmp['A']) / _acmp['A'].abs() * 100

    fig, axes = plt.subplots(1, 3, figsize=(19, 5))
    fig.suptitle(f'Per-Aisle Mean Task Duration  [{_name}]', fontsize=13, fontweight='bold')
    for _v, _lbl, _c in [(_acmp['A'], 'Uniform (A)',   _A_COL),
                          (_acmp['B'], 'Load-Min (B)', _B_COL),
                          (_acmp['C'], 'Load-Max (C)', _C_COL)]:
        axes[0].hist(_v, bins=50, color=_c, alpha=0.50, edgecolor='white', label=f'{_lbl}  μ={_v.mean():.1f}')
    axes[0].set_xlabel('Per-aisle mean task duration', fontsize=10);  axes[0].set_ylabel('Aisle count', fontsize=10)
    axes[0].set_title('Distribution of aisle mean durations', fontsize=10)
    axes[0].legend(fontsize=9);  axes[0].grid(axis='y', alpha=0.3)
    for _v, _lbl, _c in [(np.sort(_acmp['A'].values), 'Uniform (A)',   _A_COL),
                          (np.sort(_acmp['B'].values), 'Load-Min (B)', _B_COL),
                          (np.sort(_acmp['C'].values), 'Load-Max (C)', _C_COL)]:
        axes[1].plot(_v, np.arange(1, len(_v)+1)/len(_v), color=_c, lw=2, label=_lbl)
    axes[1].set_xlabel('Per-aisle mean task duration', fontsize=10);  axes[1].set_ylabel('Cumulative fraction', fontsize=10)
    axes[1].set_title('CDF of per-aisle mean duration', fontsize=10)
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    axes[1].legend(fontsize=9);  axes[1].grid(alpha=0.3)
    _dBv = _acmp['dB'].values;  _dCv = _acmp['dC'].values
    axes[2].hist(_dBv, bins=50, color=_B_COL, alpha=0.55, edgecolor='white', label=f'B vs A  mean={_dBv.mean():.2f}%')
    axes[2].hist(_dCv, bins=50, color=_C_COL, alpha=0.55, edgecolor='white', label=f'C vs A  mean={_dCv.mean():.2f}%')
    axes[2].axvline(0,           color='black', lw=1.5, linestyle='--')
    axes[2].axvline(_dBv.mean(), color=_B_COL,  lw=2,   linestyle='--')
    axes[2].axvline(_dCv.mean(), color=_C_COL,  lw=2,   linestyle='--')
    axes[2].set_xlabel('Δ (X − A) / A  %', fontsize=10);  axes[2].set_ylabel('Aisle count', fontsize=10)
    axes[2].set_title('Per-aisle % duration change (vs Uniform A)', fontsize=10)
    axes[2].xaxis.set_major_formatter(mticker.PercentFormatter())
    axes[2].legend(fontsize=9);  axes[2].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, 'plot7_per_aisle.png'), dpi=150, bbox_inches='tight')
    plt.show()

    _imp_B = (_acmp['dB'] < 0).sum();  _imp_C = (_acmp['dC'] > 0).sum()
    print(f'  Aisles faster with B: {_imp_B}/{len(_acmp)} '
          f' slower with C: {_imp_C}/{len(_acmp)}')
    print(f'  Mean delta  B: {_dBv.mean():.2f}%   C: {_dCv.mean():.2f}%')
    print(f'  Saved → {RUN_DIR}')

print(f'\nAll {len(REGRESSION_CONFIGS)} config(s) complete.  Root: {BASE_DIR}')


  Config : baseline
  Dir    : c:\Users\myfir\Code and Repos\Inventory_Location_Optimizer\Optimization\comparison_20260528_075855\baseline
  w=1.1  v=0.001  i=1.0  c=10.0
  Stocking B (load-min)... 21.4s  (55,437 bins filled)
  Stocking C (load-max)... 20.5s  (55,449 bins filled)
